## Data Preprocessing for NanoTabPFN


In [ ]:
import pandas as pd
from pathlib import Path
from scaling.tfm_utils import interpolate_all_runs

In [ ]:
# Data Loading and Visualization
base_figure_path = Path.cwd() / "figures" / "nanotabpfn"
base_figure_path.mkdir(parents=True, exist_ok=True)
base_output_path = Path.cwd() / "output" / "nanotabpfn"
base_output_path.mkdir(parents=True, exist_ok=True)
s1 = pd.read_pickle('../data/run_summary_s1_mm.pickle.xz', compression='xz')
s1_1 = pd.read_pickle('../data/run_summary_s1.1_mm.pickle.xz', compression='xz')
s1_2 = pd.read_pickle('../data/run_summary_s1.2_mm.pickle.xz', compression='xz')
s2 = pd.read_pickle('../data/run_summary_s2_mm.pickle.xz', compression='xz')
s3 = pd.read_pickle('../data/run_summary_s3_mm.pickle.xz', compression='xz')
s4 = pd.read_pickle('../data/run_summary_s4_mm.pickle.xz', compression='xz')
s5_1 = pd.read_pickle('../data/run_summary_s5.1.pickle.xz', compression='xz')
s5_2 = pd.read_pickle('../data/run_summary_s5.2.pickle.xz', compression='xz')
s5_3 = pd.read_pickle('../data/run_summary_s5.3.pickle.xz', compression='xz')

s1 = pd.concat([s1, s1_1, s1_2, s2, s3, s4, s5_1, s5_2, s5_3], ignore_index=True)
s1.head(10)

In [ ]:
s1['total_cells'].map(type).value_counts()

In [ ]:
config_columns = [col for col in s1.columns if col.startswith('config/')]
# drop constant columns
constant_columns = [col for col in config_columns if s1[col].nunique() == 1]
config_columns = [col for col in config_columns if col not in constant_columns]
print("config columns: ", config_columns)
loss_columns = ['real_data/nll', 'real_data/roc_auc', 'val/val_loss', 'val/small/val_loss', 'val/medium/val_loss']
# per-dataset real data metrics, e.g. 'real_data/splice/roc_auc' (skips 'real_data/inference_time' etc.)
real_data_dataset_columns = [col for col in s1.columns if col.startswith('real_data/') and col.count('/') == 2]
loss_columns_extended = ['real_data/nll', 'real_data/roc_auc', 'real_data/accuracy', 'real_data/balanced_accuracy',
                         'val/val_loss', 'val/small/val_loss', 'val/medium/val_loss'] + real_data_dataset_columns
every_step_columns = ['total_cells', "flops/total_mm"]
df = None
for loss_column in loss_columns_extended:
    next_df = pd.DataFrame(pd.concat(s1[loss_column].to_list(), keys=s1.index))
    if df is None:
        df = next_df
    else:
        df = df.join(next_df, how='outer', validate='one_to_one')

for every_step_column in every_step_columns:
    # key by the surviving runs' own index, otherwise a run without this column shifts all later runs by one
    every_step_series = s1[every_step_column].dropna()
    next_df = pd.DataFrame(pd.concat(every_step_series.to_list(), keys=every_step_series.index))
    df = df.join(next_df, how='left', validate='one_to_one')

df = df.join(s1[config_columns + ["parameters", "config_path"]], how='left', on=df.index.get_level_values(0)).drop(
    columns="key_0")

df['total_flops'] = df["flops/total_mm"]
df.head()

In [ ]:

value_columns = loss_columns_extended + ["total_cells"]
constant_columns = config_columns + ["parameters", "config_path"]
df = interpolate_all_runs(df, value_columns, constant_columns).set_index(["run", "grid_idx"])

print(f"{len(df)} grid points, {df['interpolated'].mean():.1%} of them interpolated")
print("share interpolated by model size:")
print(df.groupby("parameters")["interpolated"].mean().round(2).to_string())
df.head()

In [ ]:
df.drop(columns=["interpolated", "flops_deviation_dex"], inplace=True)
df.to_csv("../data/nanotabpfn_data_preprocessed.csv", index=True)